In [1]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_excel("Cleaned_Dataset.xlsx")

# Display the first 5 rows
print("First 5 rows:")
display(df.head())

# Display all column names
print("\nColumn names:")
print(df.columns.tolist())

# Display number of rows and columns
print("\nDataset shape:")
print(df.shape)

# Display data types
print("\nData types:")
print(df.dtypes)

First 5 rows:


,Order_ID,Order_Date,Customer_ID,Product,Category,Region,Quantity,Revenue,Profit,Invalid_Date_Flag,Negative_Quantity_Flag,Negative_Profit_Flag,Month,Year,Customer_Segment
0,1001,2026-01-06,C001,Laptop,Electronics,Delhi,1.0,65000.0,8000.0,False,False,False,1.0,2026.0,New Customer
1,1002,2026-03-06,C002,Phone,Electronics,Delhi,2.0,50000.0,6000.0,False,False,False,3.0,2026.0,Repeat Customer
2,1003,NaT,C003,Headphones,Electronics,Mumbai,1.0,5000.0,1200.0,True,False,False,NaN,NaN,Repeat Customer
3,1004,2026-05-06,C004,Chair,Furniture,Bangalore,-3.0,12000.0,2500.0,False,True,False,5.0,2026.0,Repeat Customer
4,1005,2026-03-07,C314,Vacuum Cleaner,Home,Delhi,NaN,8506.0,1455.0,False,False,False,3.0,2026.0,Repeat Customer



Column names:
['Order_ID', 'Order_Date', 'Customer_ID', 'Product', 'Category', 'Region', 'Quantity', 'Revenue', 'Profit', 'Invalid_Date_Flag', 'Negative_Quantity_Flag', 'Negative_Profit_Flag', 'Month', 'Year', 'Customer_Segment']

Dataset shape:
(1977, 15)

Data types:
Order_ID                           int64
Order_Date                datetime64[ns]
Customer_ID                       object
Product                           object
Category                          object
Region                            object
Quantity                         float64
Revenue                          float64
Profit                           float64
Invalid_Date_Flag                   bool
Negative_Quantity_Flag              bool
Negative_Profit_Flag                bool
Month                            float64
Year                             float64
Customer_Segment                  object
dtype: object


In [2]:
# Create customer-level data

# Make sure Order_Date is in proper date format
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

# Group transactions by customer
customer_df = df.groupby("Customer_ID").agg(
    Order_Count=("Order_ID", "count"),
    Total_Revenue=("Revenue", "sum"),
    Avg_Order_Value=("Revenue", "mean"),
    First_Purchase=("Order_Date", "min"),
    Last_Purchase=("Order_Date", "max")
).reset_index()

# Display the first 10 customers
print("First 10 customers:")
display(customer_df.head(10))

# Display number of customers
print("\nNumber of customers:", len(customer_df))

# Display column names
print("\nCustomer-level columns:")
print(customer_df.columns.tolist())

First 10 customers:


,Customer_ID,Order_Count,Total_Revenue,Avg_Order_Value,First_Purchase,Last_Purchase
0,C001,1,65000.0,65000.000000,2026-01-06,2026-01-06
1,C002,9,256033.0,28448.111111,2026-02-17,2026-12-07
2,C003,3,82737.0,27579.000000,2026-02-03,2026-06-06
3,C004,5,79072.0,15814.400000,2026-03-01,2026-06-07
4,C005,2,49630.0,24815.000000,2026-04-07,2026-12-08
5,C006,3,228080.0,76026.666667,2026-08-19,2026-11-01
6,C007,3,99452.0,33150.666667,2026-04-19,2026-07-29
7,C008,1,76659.0,76659.000000,2026-01-04,2026-01-04
8,C009,3,143415.0,47805.000000,2026-04-25,2026-09-05
9,C010,5,114185.0,22837.000000,2026-01-03,2026-12-07



Number of customers: 415

Customer-level columns:
['Customer_ID', 'Order_Count', 'Total_Revenue', 'Avg_Order_Value', 'First_Purchase', 'Last_Purchase']


In [3]:
# Create RFM features

# Use the latest valid order date as the reference date
reference_date = df["Order_Date"].max()

# Calculate RFM features
customer_df["Recency"] = (
    reference_date - customer_df["Last_Purchase"]
).dt.days

customer_df["Frequency"] = customer_df["Order_Count"]

customer_df["Monetary"] = customer_df["Total_Revenue"]

# Display the first 10 customers
print("RFM Customer Data:")
display(customer_df.head(10))

# Display the reference date
print("\nReference Date:", reference_date)

RFM Customer Data:


,Customer_ID,Order_Count,Total_Revenue,Avg_Order_Value,First_Purchase,Last_Purchase,Recency,Frequency,Monetary
0,C001,1,65000.0,65000.000000,2026-01-06,2026-01-06,336.0,1,65000.0
1,C002,9,256033.0,28448.111111,2026-02-17,2026-12-07,1.0,9,256033.0
2,C003,3,82737.0,27579.000000,2026-02-03,2026-06-06,185.0,3,82737.0
3,C004,5,79072.0,15814.400000,2026-03-01,2026-06-07,184.0,5,79072.0
4,C005,2,49630.0,24815.000000,2026-04-07,2026-12-08,0.0,2,49630.0
5,C006,3,228080.0,76026.666667,2026-08-19,2026-11-01,37.0,3,228080.0
6,C007,3,99452.0,33150.666667,2026-04-19,2026-07-29,132.0,3,99452.0
7,C008,1,76659.0,76659.000000,2026-01-04,2026-01-04,338.0,1,76659.0
8,C009,3,143415.0,47805.000000,2026-04-25,2026-09-05,94.0,3,143415.0
9,C010,5,114185.0,22837.000000,2026-01-03,2026-12-07,1.0,5,114185.0



Reference Date: 2026-12-08 00:00:00


In [4]:
# Check the available date range

print("Earliest Order Date:", df["Order_Date"].min())
print("Latest Order Date:", df["Order_Date"].max())

Earliest Order Date: 2026-01-01 00:00:00
Latest Order Date: 2026-12-08 00:00:00


In [5]:
# Proposed historical cutoff for leakage-safe churn modeling

cutoff_date = pd.Timestamp("2026-06-11")
observation_end = df["Order_Date"].max()

print("Historical period:")
print("Start:", df["Order_Date"].min())
print("End:", cutoff_date)

print("\nFuture observation period:")
print("Start:", cutoff_date + pd.Timedelta(days=1))
print("End:", observation_end)

print("\nFuture observation days:",
      (observation_end - cutoff_date).days)

Historical period:
Start: 2026-01-01 00:00:00
End: 2026-06-11 00:00:00

Future observation period:
Start: 2026-06-12 00:00:00
End: 2026-12-08 00:00:00

Future observation days: 180


In [6]:
# Create historical transaction data
historical_df = df[
    df["Order_Date"] <= cutoff_date
].copy()

# Create customer-level historical features
historical_customers = historical_df.groupby("Customer_ID").agg(
    Order_Count=("Order_ID", "count"),
    Total_Revenue=("Revenue", "sum"),
    Avg_Order_Value=("Revenue", "mean"),
    First_Purchase=("Order_Date", "min"),
    Last_Purchase=("Order_Date", "max")
).reset_index()

# Calculate historical RFM
historical_customers["Recency"] = (
    cutoff_date - historical_customers["Last_Purchase"]
).dt.days

historical_customers["Frequency"] = historical_customers["Order_Count"]

historical_customers["Monetary"] = historical_customers["Total_Revenue"]

# Display results
print("Historical customer dataset:")
display(historical_customers.head(10))

print("\nNumber of historical customers:", len(historical_customers))

print("\nHistorical columns:")
print(historical_customers.columns.tolist())

Historical customer dataset:


,Customer_ID,Order_Count,Total_Revenue,Avg_Order_Value,First_Purchase,Last_Purchase,Recency,Frequency,Monetary
0,C001,1,65000.0,65000.00,2026-01-06,2026-01-06,156,1,65000.0
1,C002,5,184795.0,36959.00,2026-02-17,2026-06-08,3,5,184795.0
2,C003,2,77737.0,38868.50,2026-02-03,2026-06-06,5,2,77737.0
3,C004,5,79072.0,15814.40,2026-03-01,2026-06-07,4,5,79072.0
4,C005,1,13363.0,13363.00,2026-04-07,2026-04-07,65,1,13363.0
5,C007,1,17340.0,17340.00,2026-04-19,2026-04-19,53,1,17340.0
6,C008,1,76659.0,76659.00,2026-01-04,2026-01-04,158,1,76659.0
7,C009,1,122420.0,122420.00,2026-04-25,2026-04-25,47,1,122420.0
8,C010,2,26508.0,13254.00,2026-01-03,2026-02-03,128,2,26508.0
9,C012,13,606585.0,50548.75,2026-02-27,2026-06-06,5,13,606585.0



Number of historical customers: 364

Historical columns:
['Customer_ID', 'Order_Count', 'Total_Revenue', 'Avg_Order_Value', 'First_Purchase', 'Last_Purchase', 'Recency', 'Frequency', 'Monetary']


In [7]:
# Create future purchase information

future_df = df[
    (df["Order_Date"] > cutoff_date) &
    (df["Order_Date"] <= observation_end)
].copy()

# Find each customer's most recent purchase in the future period
future_purchases = future_df.groupby("Customer_ID").agg(
    Future_Last_Purchase=("Order_Date", "max")
).reset_index()

# Merge future purchase information with historical customers
model_df = historical_customers.merge(
    future_purchases,
    on="Customer_ID",
    how="left"
)

# Create churn target
# Churned = no purchase during the 180-day future observation period
model_df["Churn_Status"] = model_df["Future_Last_Purchase"].isna().astype(int)

# Display results
print("Customer churn dataset:")
display(model_df.head(10))

print("\nChurn counts:")
print(model_df["Churn_Status"].value_counts())

print("\nChurn percentages:")
print(model_df["Churn_Status"].value_counts(normalize=True) * 100)

Customer churn dataset:


,Customer_ID,Order_Count,Total_Revenue,Avg_Order_Value,First_Purchase,Last_Purchase,Recency,Frequency,Monetary,Future_Last_Purchase,Churn_Status
0,C001,1,65000.0,65000.00,2026-01-06,2026-01-06,156,1,65000.0,NaT,1
1,C002,5,184795.0,36959.00,2026-02-17,2026-06-08,3,5,184795.0,2026-12-07,0
2,C003,2,77737.0,38868.50,2026-02-03,2026-06-06,5,2,77737.0,NaT,1
3,C004,5,79072.0,15814.40,2026-03-01,2026-06-07,4,5,79072.0,NaT,1
4,C005,1,13363.0,13363.00,2026-04-07,2026-04-07,65,1,13363.0,2026-12-08,0
5,C007,1,17340.0,17340.00,2026-04-19,2026-04-19,53,1,17340.0,2026-07-29,0
6,C008,1,76659.0,76659.00,2026-01-04,2026-01-04,158,1,76659.0,NaT,1
7,C009,1,122420.0,122420.00,2026-04-25,2026-04-25,47,1,122420.0,2026-09-05,0
8,C010,2,26508.0,13254.00,2026-01-03,2026-02-03,128,2,26508.0,2026-12-07,0
9,C012,13,606585.0,50548.75,2026-02-27,2026-06-06,5,13,606585.0,2026-11-08,0



Churn counts:
Churn_Status
0    266
1     98
Name: count, dtype: int64

Churn percentages:
Churn_Status
0    73.076923
1    26.923077
Name: proportion, dtype: float64


In [8]:
# Check the final modeling dataset

print("Model dataset shape:", model_df.shape)

print("\nModel columns:")
print(model_df.columns.tolist())

print("\nMissing values in model dataset:")
print(model_df.isnull().sum())

Model dataset shape: (364, 11)

Model columns:
['Customer_ID', 'Order_Count', 'Total_Revenue', 'Avg_Order_Value', 'First_Purchase', 'Last_Purchase', 'Recency', 'Frequency', 'Monetary', 'Future_Last_Purchase', 'Churn_Status']

Missing values in model dataset:
Customer_ID              0
Order_Count              0
Total_Revenue            0
Avg_Order_Value          0
First_Purchase           0
Last_Purchase            0
Recency                  0
Frequency                0
Monetary                 0
Future_Last_Purchase    98
Churn_Status             0
dtype: int64


In [9]:
# Select leakage-safe prediction features

features = [
    "Recency",
    "Frequency",
    "Monetary",
    "Avg_Order_Value"
]

target = "Churn_Status"

X = model_df[features].copy()
y = model_df[target].copy()

print("Prediction features:")
print(X.columns.tolist())

print("\nTarget:")
print(target)

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

print("\nMissing values in features:")
print(X.isnull().sum())

print("\nTarget distribution:")
print(y.value_counts())

Prediction features:
['Recency', 'Frequency', 'Monetary', 'Avg_Order_Value']

Target:
Churn_Status

Feature shape: (364, 4)
Target shape: (364,)

Missing values in features:
Recency            0
Frequency          0
Monetary           0
Avg_Order_Value    0
dtype: int64

Target distribution:
Churn_Status
0    266
1     98
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training set: (291, 4)
Testing set: (73, 4)

Training target distribution:
Churn_Status
0    213
1     78
Name: count, dtype: int64

Testing target distribution:
Churn_Status
0    53
1    20
Name: count, dtype: int64


In [11]:
from sklearn.linear_model import LogisticRegression

# Create and train the Logistic Regression model
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

# Make predictions on the test data
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Model Evaluation")
print("-----------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Evaluation
-----------------
Accuracy : 0.726027397260274
Precision: 0.0
Recall   : 0.0

Classification Report:
              precision    recall  f1-score   support

           0       0.73      1.00      0.84        53
           1       0.00      0.00      0.00        20

    accuracy                           0.73        73
   macro avg       0.36      0.50      0.42        73
weighted avg       0.53      0.73      0.61        73



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scale the prediction features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression on scaled features
scaled_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

scaled_model.fit(X_train_scaled, y_train)

print("Scaled Logistic Regression model trained successfully.")

Scaled Logistic Regression model trained successfully.


In [14]:
# Evaluate the scaled Logistic Regression model

y_pred_scaled = scaled_model.predict(X_test_scaled)

accuracy_scaled = accuracy_score(y_test, y_pred_scaled)
precision_scaled = precision_score(y_test, y_pred_scaled, zero_division=0)
recall_scaled = recall_score(y_test, y_pred_scaled, zero_division=0)

print("Scaled Model Evaluation")
print("-----------------------")
print("Accuracy :", accuracy_scaled)
print("Precision:", precision_scaled)
print("Recall   :", recall_scaled)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_scaled,
    zero_division=0
))

Scaled Model Evaluation
-----------------------
Accuracy : 0.726027397260274
Precision: 0.0
Recall   : 0.0

Classification Report:
              precision    recall  f1-score   support

           0       0.73      1.00      0.84        53
           1       0.00      0.00      0.00        20

    accuracy                           0.73        73
   macro avg       0.36      0.50      0.42        73
weighted avg       0.53      0.73      0.61        73



In [15]:
# Train a balanced Logistic Regression model

balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

balanced_model.fit(X_train_scaled, y_train)

print("Balanced Logistic Regression model trained successfully.")

Balanced Logistic Regression model trained successfully.


In [16]:
# Evaluate the balanced Logistic Regression model

y_pred_balanced = balanced_model.predict(X_test_scaled)

accuracy_balanced = accuracy_score(y_test, y_pred_balanced)
precision_balanced = precision_score(
    y_test, y_pred_balanced, zero_division=0
)
recall_balanced = recall_score(
    y_test, y_pred_balanced, zero_division=0
)

print("Balanced Model Evaluation")
print("-------------------------")
print("Accuracy :", accuracy_balanced)
print("Precision:", precision_balanced)
print("Recall   :", recall_balanced)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_balanced,
    zero_division=0
))

Balanced Model Evaluation
-------------------------
Accuracy : 0.6301369863013698
Precision: 0.4186046511627907
Recall   : 0.9

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.53      0.67        53
           1       0.42      0.90      0.57        20

    accuracy                           0.63        73
   macro avg       0.68      0.71      0.62        73
weighted avg       0.79      0.63      0.65        73



In [17]:
# Calculate churn probabilities for all customers

model_df["Churn_Probability"] = balanced_model.predict_proba(
    scaler.transform(model_df[features])
)[:, 1]

# Display customers with their churn probability
risk_df = model_df[
    ["Customer_ID", "Recency", "Frequency", "Monetary",
     "Avg_Order_Value", "Churn_Status", "Churn_Probability"]
].copy()

risk_df = risk_df.sort_values(
    "Churn_Probability",
    ascending=False
)

print("Top customers by predicted churn probability:")
display(risk_df.head(10))

Top customers by predicted churn probability:


,Customer_ID,Recency,Frequency,Monetary,Avg_Order_Value,Churn_Status,Churn_Probability
346,C456,161,1,9576.0,9576.0,0,0.723789
80,C106,160,1,4925.0,4925.0,1,0.723203
202,C264,158,1,17470.0,17470.0,1,0.720680
201,C263,157,1,5729.0,5729.0,1,0.720528
72,C091,159,1,46493.0,46493.0,1,0.719764
176,C231,155,1,7578.0,7578.0,0,0.718654
252,C323,155,1,9028.0,9028.0,0,0.718564
121,C154,156,1,36930.0,36930.0,0,0.717714
6,C008,158,1,76659.0,76659.0,1,0.717010
0,C001,156,1,65000.0,65000.0,1,0.715966


In [18]:
# Create customer risk levels

def assign_risk(probability):
    if probability >= 0.70:
        return "High Risk"
    elif probability >= 0.40:
        return "Medium Risk"
    else:
        return "Low Risk"

risk_df["Risk_Level"] = risk_df["Churn_Probability"].apply(assign_risk)

# Display risk distribution
print("Customer Risk Distribution:")
print(risk_df["Risk_Level"].value_counts())

print("\nRisk Distribution (%):")
print(
    risk_df["Risk_Level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTop High-Risk Customers:")
display(
    risk_df[risk_df["Risk_Level"] == "High Risk"].head(10)
)

Customer Risk Distribution:
Risk_Level
Medium Risk    234
Low Risk       108
High Risk       22
Name: count, dtype: int64

Risk Distribution (%):
Risk_Level
Medium Risk    64.29
Low Risk       29.67
High Risk       6.04
Name: proportion, dtype: float64

Top High-Risk Customers:


,Customer_ID,Recency,Frequency,Monetary,Avg_Order_Value,Churn_Status,Churn_Probability,Risk_Level
346,C456,161,1,9576.0,9576.0,0,0.723789,High Risk
80,C106,160,1,4925.0,4925.0,1,0.723203,High Risk
202,C264,158,1,17470.0,17470.0,1,0.720680,High Risk
201,C263,157,1,5729.0,5729.0,1,0.720528,High Risk
72,C091,159,1,46493.0,46493.0,1,0.719764,High Risk
176,C231,155,1,7578.0,7578.0,0,0.718654,High Risk
252,C323,155,1,9028.0,9028.0,0,0.718564,High Risk
121,C154,156,1,36930.0,36930.0,0,0.717714,High Risk
6,C008,158,1,76659.0,76659.0,1,0.717010,High Risk
0,C001,156,1,65000.0,65000.0,1,0.715966,High Risk


In [19]:
# Define business actions based on customer risk level

def recommend_action(risk_level):
    if risk_level == "High Risk":
        return "Immediate retention outreach, personalized offer, and follow-up"
    elif risk_level == "Medium Risk":
        return "Targeted engagement campaign and personalized recommendations"
    else:
        return "Regular engagement, loyalty offers, and relationship maintenance"

risk_df["Recommended_Action"] = risk_df["Risk_Level"].apply(
    recommend_action
)

# Display customer risk and recommended actions
print("Customer Risk and Recommended Actions:")
display(
    risk_df[
        [
            "Customer_ID",
            "Churn_Probability",
            "Risk_Level",
            "Recommended_Action"
        ]
    ].head(15)
)

Customer Risk and Recommended Actions:


,Customer_ID,Churn_Probability,Risk_Level,Recommended_Action
346,C456,0.723789,High Risk,"Immediate retention outreach, personalized off..."
80,C106,0.723203,High Risk,"Immediate retention outreach, personalized off..."
202,C264,0.720680,High Risk,"Immediate retention outreach, personalized off..."
201,C263,0.720528,High Risk,"Immediate retention outreach, personalized off..."
72,C091,0.719764,High Risk,"Immediate retention outreach, personalized off..."
176,C231,0.718654,High Risk,"Immediate retention outreach, personalized off..."
252,C323,0.718564,High Risk,"Immediate retention outreach, personalized off..."
121,C154,0.717714,High Risk,"Immediate retention outreach, personalized off..."
6,C008,0.717010,High Risk,"Immediate retention outreach, personalized off..."
0,C001,0.715966,High Risk,"Immediate retention outreach, personalized off..."


In [20]:
# Save the final customer risk analysis

risk_df.to_csv(
    "MC3_Customer_Churn_Risk_Analysis.csv",
    index=False
)

print("File saved successfully:")
print("MC3_Customer_Churn_Risk_Analysis.csv")

File saved successfully:
MC3_Customer_Churn_Risk_Analysis.csv


In [21]:
from google.colab import files

files.download("MC3_Customer_Churn_Risk_Analysis.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>